# Express `req` and `res` Objects

In Express.js, the Request (`req`) and Response (`res`) objects are the core building blocks used to handle client-server communication. They are passed as arguments to your route handlers and middleware functions.

Both are extensions of Node's built-in `http.IncomingMessage` and `http.ServerResponse`. Express adds convenience properties and methods on top, which is why anything available in raw Node is also available here.

## 1. The Request Object (`req`)

The `req` object represents the incoming HTTP request from the client (a browser, mobile app, or another service). It contains all the data sent by the client.

### Core Properties

| Property | What it holds | Example |
| --- | --- | --- |
| `req.params` | Route variables from dynamic URLs | Path `/user/:id`, visiting `/user/42` → `req.params.id === "42"` |
| `req.query` | Query string values after `?` | `/search?term=books` → `req.query.term === "books"` |
| `req.body` | Submitted payload (form or JSON) | Requires `express.json()` or `express.urlencoded()` |
| `req.headers` | HTTP headers sent by the client | `req.headers['authorization']` |
| `req.method` | The HTTP verb | `GET`, `POST`, `PUT`, `DELETE` |
| `req.path` | Path portion of the URL | `/users/42` |
| `req.ip` | Remote IP address of the client | `'::1'` locally |
| `req.cookies` | Parsed cookies | Requires the `cookie-parser` middleware |

### Two Things Worth Knowing

**Everything arrives as a string.** `req.params.id` is `"42"`, not `42`. Comparing with `===` against a number silently fails. Convert explicitly (`Number(req.params.id)`) and validate — `Number("abc")` gives `NaN`, not an error.

**`params` vs `query` vs `body`** — the distinction is about *purpose*, not just syntax:

- **`params`** identifies *which* resource: `/users/42`, `/posts/99/comments`. Part of the route definition.
- **`query`** modifies *how* you want it: `?page=2&sort=desc&limit=20`. Optional filters and options.
- **`body`** carries the *data being sent*: the new user, the updated record. Used with `POST`, `PUT`, `PATCH`.

Getting `undefined` from `req.body` almost always means `express.json()` wasn't registered, or was registered *after* the route.

## 2. The Response Object (`res`)

The `res` object represents the HTTP response the server sends back. You use its methods to set headers, status codes, and transmit data.

### Core Methods

| Method | Purpose |
| --- | --- |
| `res.send()` | Sends a response, auto-setting `Content-Type` based on string, object, or buffer |
| `res.json()` | Sends JSON — the standard for REST APIs |
| `res.status()` | Sets the status code; chainable |
| `res.redirect()` | Redirects the browser to another URL |
| `res.cookie()` | Sets a cookie on the client |
| `res.sendFile()` | Streams a file from disk |
| `res.set()` | Sets a response header manually |
| `res.end()` | Ends the response with no body |
| `res.locals` | Object scoped to the current request/response cycle, for passing data between middleware |

### The One Rule: Exactly One Response

Each request must be terminated by **one** response. Send twice and you get `ERR_HTTP_HEADERS_SENT`; send zero times and the client hangs until it times out.

This is why validation guards use `return`:

```javascript
if (!newUser.name) {
    return res.status(400).json({ error: 'Name is required' });
    // Without `return`, execution continues and tries to respond again
}
```

`res.status()` alone does **not** send anything — it only sets the code. It must be chained with `send()`, `json()`, or `end()`.

### Common Status Codes

| Code | Meaning | When to use |
| --- | --- | --- |
| `200` | OK | Successful `GET`, `PUT`, `PATCH` |
| `201` | Created | Successful `POST` that created a resource |
| `204` | No Content | Successful `DELETE`, nothing to return |
| `400` | Bad Request | Validation failed, malformed input |
| `401` | Unauthorized | Not logged in / missing credentials |
| `403` | Forbidden | Logged in, but not allowed |
| `404` | Not Found | Resource doesn't exist |
| `409` | Conflict | Duplicate resource, version conflict |
| `500` | Internal Server Error | Something broke on your side |

The distinction between `401` and `403` trips people up: `401` means "I don't know who you are," `403` means "I know who you are and you can't do this."

## Comprehensive Code Example

```javascript
const express = require('express');
const app = express();

// Middleware to parse incoming JSON bodies (populates req.body)
app.use(express.json());

// 1. Using req.query and res.send
app.get('/search', (req, res) => {
    const searchTerm = req.query.q;
    res.status(200).send(`You searched for: ${searchTerm}`);
});

// 2. Using req.params and res.json
app.get('/users/:id', (req, res) => {
    const userId = req.params.id;

    // Simulating a quick data lookup
    if (userId === '1') {
        res.status(200).json({ id: 1, name: 'Alice' });
    } else {
        res.status(404).json({ error: 'User not found' });
    }
});

// 3. Using req.body
app.post('/users', (req, res) => {
    const newUser = req.body;

    if (!newUser.name) {
        return res.status(400).json({ error: 'Name field is required' });
    }

    res.status(201).json({
        message: 'User created successfully',
        data: newUser
    });
});

app.listen(3000, () => console.log('Server running on port 3000'));
```

## Passing Data Between Middleware

`res.locals` is the clean way to hand data from one middleware to the next within a single request:

```javascript
// Auth middleware resolves the user once
app.use((req, res, next) => {
    res.locals.user = lookupUserFromToken(req.headers.authorization);
    next();
});

// Any later handler can read it
app.get('/profile', (req, res) => {
    res.json(res.locals.user);
});
```

Attaching the value to `req` instead (`req.user = ...`) is also extremely common in real codebases — both work. `res.locals` has the advantage of being automatically available to template engines when rendering views.

Because `res.locals` is created fresh for each request, there's no risk of leaking one user's data into another's response.

## References

- [Express API — Request](https://expressjs.com/en/5x/api/request.html)
- [Express API — Response](https://expressjs.com/en/5x/api/response/)